# Búsqueda distribuida de hiperparámetros: secuencial vs. paralelo vs. distribuido

**Materia:** Cómputo de Alto Rendimiento · **Grupo:** Ingeniería en Datos e Inteligencia Artificial, 7.º cuatrimestre
**Evidencia de producto P2** · Proyecto 2: *Búsqueda distribuida de hiperparámetros (básico)*

**Equipo:** _[nombres de los integrantes]_ · **Fecha:** _[fecha de entrega]_

---

## Resumen

Se diseña, implementa y evalúa una búsqueda de hiperparámetros para un **Random Forest** sobre el dataset **Covertype (UCI)**, con validación cruzada estratificada. Se comparan: (1) una versión **secuencial** con un ciclo `for`; (2) una versión de **memoria compartida/procesos locales** con `multiprocessing.Pool` (reparto **estático** vs. **dinámico** con `chunksize`) y `joblib`; y (3) una versión **distribuida** con `mpi4py` en patrón **maestro–trabajador** (con Ray como alternativa opcional). Cada configuración se repite varias veces y se reporta **media ± desviación estándar** de tiempo, *speedup*, eficiencia, desbalance de carga y fracción paralela estimada con la **Ley de Amdahl**. Como reto extra se grafica el tiempo de cada tarea por trabajador y el efecto del desbalance en la eficiencia.

> Las cifras se generan al ejecutar el notebook en la máquina del equipo; las secciones de resultados y análisis se llenan con las mediciones reales.

## 1. Introducción y objetivo

Ajustar hiperparámetros es una de las tareas más costosas del aprendizaje automático: para cada combinación hay que entrenar el modelo varias veces (una por *fold*). Con cientos de combinaciones, el tiempo total crece rápidamente.

**Objetivo.** Diseñar, implementar y evaluar una solución paralela a un problema real de datos/IA, demostrando con mediciones cuánto mejora el rendimiento respecto a una versión secuencial y explicando por qué.

**Preguntas que guían el experimento**

1. ¿Cuánto *speedup* y qué eficiencia se obtienen con 1, 2, 4, 8 y 16 procesos?
2. ¿Qué diferencia hay entre repartir las combinaciones de forma **estática** y de forma **dinámica**?
3. ¿Cómo se compara un esquema local (`Pool`) con uno de paso de mensajes (MPI, maestro–trabajador)?
4. ¿Cómo afecta el **desbalance de carga** a la eficiencia y por qué el *speedup* no llega al ideal?

**Estructura del repositorio** (cada módulo se muestra completo más adelante):

```
hpc-p2-hiperparametros/
├── comun/            worker.py (función de trabajo compartida), utils_hpo.py (configuración y registro)
├── secuencial/       secuencial.py
├── paralelo_pool/    pool_hpo.py      (memoria compartida / procesos locales: Pool y joblib)
├── paralelo_mpi/     mpi_hpo.py       (memoria distribuida: mpi4py maestro–trabajador)
├── paralelo_ray/     ray_hpo.py       (alternativa opcional)
├── experimentos/     este notebook
├── resultados/       CSV por corrida, resumen y figuras (se generan al ejecutar)
├── README.md · requirements.txt · Dockerfile
```

## 2. Problema, paralelización y métricas

### 2.1 Un problema "embarazosamente paralelo"

Cada combinación de hiperparámetros se evalúa **sin depender de ninguna otra**: no hay comunicación entre tareas ni datos intermedios compartidos. Basta con repartir las combinaciones entre procesos y, al final, juntar los resultados y elegir la mejor. En teoría el *speedup* ideal con *p* procesos es *p*.

### 2.2 El reto real: el balance de carga

No todas las combinaciones tardan lo mismo. En un Random Forest el costo depende de `n_estimators` (casi lineal), `max_depth` (los árboles sin límite son mucho más costosos), `max_features` y `min_samples_split`/`min_samples_leaf`. Unas tareas duran fracciones de segundo y otras decenas de segundos. Con un reparto **estático** (bloques contiguos fijados de antemano) un proceso puede recibir justo las tareas pesadas mientras otros terminan pronto y quedan **ociosos**; el tiempo total lo marca el más lento. Con un reparto **dinámico** (cada proceso pide trabajo al terminar) la carga se equilibra sola, a costa de algo más de coordinación.

### 2.3 Fórmulas aplicadas

*Speedup* con $p$ procesadores, donde $T_1$ es el tiempo de la versión secuencial y $T_p$ el de la versión paralela con $p$ procesos:

$$S(p)=\frac{T_1}{T_p}$$

Eficiencia:

$$E(p)=\frac{S(p)}{p}$$

**Ley de Amdahl**, con $f$ la **fracción paralelizable** del programa:

$$S(p)\le\frac{1}{(1-f)+\frac{f}{p}}$$

Despejando $f$ a partir del *speedup* medido, se obtiene la **fracción paralela estimada** y el máximo teórico con infinitos procesos, $S_{\max}=\frac{1}{1-f}$:

$$f=\frac{1-\frac{1}{S(p)}}{1-\frac{1}{p}}$$

Además, para cuantificar el **desbalance de carga** se usa el tiempo ocupado $B_i$ de cada trabajador (suma de las duraciones de sus tareas):

$$\text{Desbalance}=\frac{\max_i B_i}{\overline{B}}\qquad(1.0=\text{perfecto})$$

y el tiempo ideal $T_{ideal}=\sum_j t_j/p$ (todas las tareas perfectamente repartidas y sin sobrecarga).

> **Nota (del enunciado):** un *speedup* bajo no reprueba el proyecto; lo que se evalúa es explicar con datos por qué no escala más (sobrecarga de comunicación, desbalance, secciones secuenciales, ancho de banda de memoria, E/S…).

## 3. Entorno y configuración del experimento

In [ ]:
import os, sys, time, math, json, platform, pickle, subprocess, shutil
from pathlib import Path
import multiprocessing as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code
import sklearn
from sklearn.model_selection import train_test_split

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# --- Rutas del repositorio (el notebook vive en experimentos/) ---
ROOT = Path.cwd().parent if Path.cwd().name == "experimentos" else Path.cwd()
assert (ROOT / "comun" / "worker.py").exists(), "Abre el notebook desde la carpeta experimentos/ del repositorio."
MODULOS = [ROOT / d for d in ("comun", "secuencial", "paralelo_pool", "paralelo_mpi", "paralelo_ray")]
for m in MODULOS:
    sys.path.insert(0, str(m))
# los procesos hijos (loky, Ray, mpiexec) también necesitan encontrar los módulos
os.environ["PYTHONPATH"] = os.pathsep.join([str(m) for m in MODULOS] + [os.environ.get("PYTHONPATH", "")])

RES = ROOT / "resultados"      # CSV por corrida, resumen y figuras
DAT = ROOT / "datos"           # dataset en caché

from worker import evaluate, init_worker, reset_data
from utils_hpo import CONFIGS, construir_tareas, costo_estimado, Registro
from secuencial import run_secuencial
from pool_hpo import run_pool, run_joblib

# ----------------------------- CONFIGURACIÓN -----------------------------
MODO = "estandar"                 # "prueba" | "estandar" | "completo"  (ver utils_hpo.CONFIGS)
USAR_DATOS_SINTETICOS = False   # solo para depurar SIN internet; para el reporte debe ser False
SEMILLA = 42
CV_FOLDS = 3
REPS = 3                      # repeticiones por configuración (el enunciado pide >= 3)
MAX_PROCS = 16                  # el enunciado pide 1, 2, 4, 8 "y más si es posible"; se recorta a los núcleos lógicos
INCLUIR_ORDENADO = True         # estrategia extra: dinámico con tareas ordenadas por costo
CORRER_JOBLIB = True            # comparación con joblib (con el máximo de procesos)
CORRER_RAY = False              # alternativa a MPI (requiere `pip install ray`)
REHACER = False                 # False: si una corrida ya está guardada en resultados/, no se repite
# --------------------------------------------------------------------------

CFG = CONFIGS[MODO]
N_CORES = os.cpu_count()
PMAX = max(1, min(MAX_PROCS, N_CORES))
P_LIST = sorted({p for p in (1, 2, 4, 8, 16, 32) if p <= PMAX} | {PMAX})

try:
    import psutil
    ram = f"{psutil.virtual_memory().total/2**30:.1f} GB"; fisicos = psutil.cpu_count(logical=False)
except Exception:
    ram, fisicos = "n/d", "n/d"
gpu = "No aplica (solo CPU)"
if shutil.which("nvidia-smi"):
    try: gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or gpu
    except Exception: pass

reg = Registro(RES)
display(Markdown(f'''
| Especificaciones del equipo de cómputo | |
|---|---|
| Sistema operativo | {platform.system()} {platform.release()} |
| Procesador | {platform.processor() or 'n/d'} |
| Núcleos lógicos / físicos | {N_CORES} / {fisicos} |
| RAM | {ram} |
| GPU | {gpu} |
| Python / scikit-learn | {platform.python_version()} / {sklearn.__version__} |

| Configuración del experimento | |
|---|---|
| Modo | **{MODO}** · muestra de {CFG['n_muestra']:,} registros |
| Procesos probados | {P_LIST} |
| Repeticiones por configuración | {REPS} |
| Validación cruzada | {CV_FOLDS} folds estratificados |
'''))

**Configuración elegida.** Se usa Random Forest porque su costo varía mucho entre combinaciones (lo que hace visible el balance de carga). Cada tarea es una combinación evaluada con validación cruzada de 3 folds y `n_jobs=1` dentro de la tarea: **el paralelismo lo aportamos nosotros entre tareas**, no scikit-learn, para que la comparación sea limpia. Las semillas fijas hacen que cada combinación sea determinista, así que todas las versiones deben producir exactamente los mismos puntajes (se verifica en la sección 6.1).

## 4. Dataset: Covertype (UCI)

Covertype describe áreas de 30 × 30 m del Roosevelt National Forest (Colorado) con **581,012 registros y 54 variables** (10 cuantitativas, 4 indicadoras de área silvestre y 40 de tipo de suelo). La variable objetivo es el **tipo de cobertura forestal** (7 clases, desbalanceadas: dominan las clases 1 y 2), por lo que además de *accuracy* se reporta **F1 macro**.

Para controlar el tiempo total se trabaja con una **muestra estratificada** (`n_muestra`) y se reserva un 20 % como conjunto de **prueba** que no participa en la búsqueda: la validación cruzada se hace solo sobre el 80 % restante.

In [ ]:
from sklearn.datasets import fetch_covtype
cov = fetch_covtype(data_home=str(DAT / "sklearn_data"))     # descarga y guarda en caché la primera vez
X_all, y_all = cov.data, cov.target                          # y en 1..7
print("Covertype completo:", X_all.shape)

n = CFG["n_muestra"]
if n is not None and n < len(X_all):
    X_all, _, y_all, _ = train_test_split(X_all, y_all, train_size=n, stratify=y_all, random_state=SEMILLA)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEMILLA)

DAT.mkdir(exist_ok=True)
DATA = str(DAT / "covtype_muestra.npz")
np.savez(DATA, X=X_tr.astype(np.float32), y=y_tr.astype(np.int8))   # cada proceso leerá este archivo

print(f"Entrenamiento/CV: {X_tr.shape}   Prueba: {X_te.shape}")
dist = pd.Series(y_tr).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(dist.index.astype(str), dist.values, color="#4c72b0")
ax.set_xlabel("Clase (tipo de cobertura)"); ax.set_ylabel("Registros"); ax.set_title("Distribución de clases (entrenamiento)")
plt.show()

TASKS = construir_tareas(CFG["grid"], DATA, CV_FOLDS, SEMILLA)
N_TAREAS = len(TASKS)
print(f"{N_TAREAS} combinaciones de hiperparámetros (tareas).")

## 5. Implementación

El código vive en módulos del repositorio (se muestran aquí completos). **Todas las versiones ejecutan exactamente la misma función `evaluate`**, para que la comparación sea justa. Está en un módulo aparte porque en Windows/macOS los procesos hijos se crean con *spawn* y no pueden importar funciones definidas dentro del notebook.

Decisiones de diseño comunes:

- Cada proceso **carga el dataset una sola vez** (`initializer`) y lo guarda en una variable global, para que la lectura de disco no se repita por tarea ni contamine las mediciones.
- Cada tarea devuelve, además de su puntaje, el **instante de inicio/fin y el PID**; con eso se construyen el diagrama de tiempos por trabajador y las métricas de desbalance.
- El tiempo de cada corrida incluye arrancar los procesos y cargar el dataset, de modo que la comparación con la versión secuencial (que también lo carga) sea equitativa.

### 5.1 Función de trabajo compartida (`comun/worker.py`)

In [ ]:
display(Code(filename=str(ROOT / "comun" / "worker.py"), language="python"))

### 5.2 Versión 1 — Secuencial (`secuencial/secuencial.py`)

Un ciclo `for` evalúa las combinaciones una por una. Su tiempo es $T_1$, la referencia de todas las mediciones.

In [ ]:
display(Code(filename=str(ROOT / "secuencial" / "secuencial.py"), language="python"))

### 5.3 Versión 2 — Memoria compartida / procesos locales (`paralelo_pool/pool_hpo.py`)

`multiprocessing.Pool` con tres estrategias de reparto:

| Estrategia | Cómo reparte | Llamada |
|---|---|---|
| **Estático** | *p* bloques contiguos del mismo tamaño, uno por proceso | `pool.map(..., chunksize=ceil(N/p))` |
| **Dinámico** | cada proceso pide la siguiente tarea al terminar la anterior | `pool.imap_unordered(..., chunksize=1)` |
| **Dinámico ordenado** *(extra)* | como el dinámico, pero entrega primero las tareas estimadas como más pesadas (heurística tipo *LPT*) | `imap_unordered` sobre la lista ordenada |

El orden de `ParameterGrid` deja `max_depth` como el parámetro que cambia más lento, así que un bloque contiguo concentra profundidades parecidas: justo el caso que perjudica al reparto estático. También se incluye la variante con `joblib` (`batch_size` grande ≈ estático, `batch_size=1` ≈ dinámico).

In [ ]:
display(Code(filename=str(ROOT / "paralelo_pool" / "pool_hpo.py"), language="python"))

### 5.4 Versión 3 — Memoria distribuida con `mpi4py` (`paralelo_mpi/mpi_hpo.py`)

Patrón **maestro–trabajador** con reparto bajo demanda: el rango 0 (maestro) no entrena, administra la cola; cada trabajador avisa que está libre (entregando su resultado) y recibe la siguiente combinación o una señal de paro. MPI se lanza con `mpiexec` como programa aparte (no como función del notebook), con *p + 1* procesos para *p* trabajadores.

In [ ]:
display(Code(filename=str(ROOT / "paralelo_mpi" / "mpi_hpo.py"), language="python"))

### 5.5 Calibración rápida

Se mide la combinación **más ligera** y la **más pesada** de la malla para anticipar el desbalance y el tiempo total.

In [ ]:
orden = sorted(range(N_TAREAS), key=lambda i: costo_estimado(TASKS[i][1]))
ligera, pesada = TASKS[orden[0]], TASKS[orden[-1]]
reset_data(); init_worker(DATA)
r_lig, r_pes = evaluate(ligera), evaluate(pesada)
print("Más ligera :", ligera[1], f"-> {r_lig['dur']:.2f} s")
print("Más pesada :", pesada[1], f"-> {r_pes['dur']:.2f} s")
print(f"Razón pesada/ligera: {r_pes['dur']/max(r_lig['dur'],1e-9):.1f}x")
est = N_TAREAS * (r_lig['dur'] + r_pes['dur']) / 2
print(f"Estimación MUY gruesa del tiempo secuencial: {est/60:.1f} min por repetición")

## 6. Experimentos

Cada configuración (versión × número de procesos) se repite `REPS` veces. Las corridas se guardan en `resultados/` (se pueden retomar si se reinicia el kernel). Los procesos probados son 1, 2, 4, 8 y 16 (según los núcleos disponibles); con `p = 1` la referencia es la versión secuencial.

### 6.1 Ejecución

In [ ]:
def correr(name, p, rep, fn):
    if reg.existe(name, p, rep) and not REHACER:
        print(f"[omitido] {name} p={p} rep={rep+1}"); return
    wall, t0, df = fn()
    reg.guardar(name, p, rep, wall, t0, df)
    print(f"{name:<24} p={p:<2} rep={rep+1}/{REPS}   {wall:8.1f} s")

# ---- Versión 1: secuencial ----
for rep in range(REPS):
    correr("secuencial", 1, rep, lambda: run_secuencial(TASKS, DATA))

T1 = np.mean([r["wall"] for r in reg.cargar() if r["name"] == "secuencial"])
n_est = 1 + (1 if INCLUIR_ORDENADO else 0) + 1
resto = REPS * T1 * sum((n_est if p > 1 else 1) / p for p in P_LIST) * (2 if CORRER_JOBLIB else 1.5)
print(f"\nT1 (media secuencial) = {T1:.1f} s. Estimación gruesa del resto del experimento: ~{resto/60:.0f} min.")

In [ ]:
# ---- Versión 2: Pool (estático / dinámico / dinámico ordenado) ----
ESTRATEGIAS = ["estatico", "dinamico"] + (["dinamico_ordenado"] if INCLUIR_ORDENADO else [])
for p in P_LIST:
    for est in ESTRATEGIAS:
        if p == 1 and est != "dinamico":
            continue            # con 1 proceso todas las estrategias son equivalentes
        for rep in range(REPS):
            correr(f"pool_{est}", p, rep, lambda: run_pool(est, p, TASKS, DATA))

In [ ]:
# ---- Comparación con joblib (solo con el máximo de procesos) ----
if CORRER_JOBLIB and PMAX > 1:
    for est in ["estatico", "dinamico"]:
        for rep in range(REPS):
            correr(f"joblib_{est}", PMAX, rep, lambda: run_joblib(est, PMAX, TASKS))

In [ ]:
# ---- Versión 3: MPI maestro-trabajador (lanzado con mpiexec) ----
def run_mpi(workers, rep):
    RES.mkdir(exist_ok=True)
    pickle.dump(TASKS, open(RES / "tasks.pkl", "wb"))
    prefijo = str(RES / f"mpi_raw_w{workers}")
    script = str(ROOT / "paralelo_mpi" / "mpi_hpo.py")
    for extra in ([], ["--oversubscribe"]):
        cmd = ["mpiexec", *extra, "-n", str(workers + 1), sys.executable, script, str(RES / "tasks.pkl"), prefijo]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            break
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise RuntimeError("Falló mpiexec (¿MPI instalado? En Windows: MS-MPI)")
    meta = json.load(open(prefijo + ".json"))
    df = pd.read_csv(prefijo + ".csv"); df["worker"] = df["rank"] - 1
    os.remove(prefijo + ".csv"); os.remove(prefijo + ".json")
    return meta["wall"], meta["t0"], df

for p in P_LIST:
    for rep in range(REPS):
        try:
            correr("mpi", p, rep, lambda: run_mpi(p, rep))
        except Exception as e:
            print("MPI no disponible en este equipo:", e); break

In [ ]:
# ---- Alternativa opcional: Ray ----
if CORRER_RAY:
    from ray_hpo import run_ray
    for p in P_LIST:
        for rep in range(REPS):
            correr("ray", p, rep, lambda: run_ray(p, TASKS))
else:
    print("Ray desactivado (CORRER_RAY = False).")

### 6.2 Verificación de correctitud

Se comprueba que **todas las versiones, en todas las repeticiones, producen los mismos puntajes** que la versión secuencial para cada combinación (la paralelización no debe cambiar el resultado), y se identifica la mejor combinación.

In [ ]:
RUNS = reg.todas()
ref = RUNS[("secuencial", 1, 0)]["df"].set_index("idx")[["acc", "f1"]].sort_index()
filas, max_diff = [], 0.0
for (name, p, rep), r in sorted(RUNS.items()):
    cur = r["df"].set_index("idx")[["acc", "f1"]].sort_index()
    ok_n = len(cur) == len(ref)
    d = float((cur - ref).abs().max().max()) if ok_n else np.inf
    max_diff = max(max_diff, d)
resumen_ok = pd.DataFrame([dict(version=n, p=p, rep=r+1, combinaciones=len(v["df"])) for (n, p, r), v in sorted(RUNS.items())])
print(f"Corridas comparadas con la secuencial: {len(RUNS)}  ·  diferencia máxima de accuracy/F1: {max_diff:.2e}")
assert max_diff < 1e-9, "Las versiones no coinciden: revisar semillas"
print("Correctitud verificada: todas las versiones coinciden con la secuencial.\n")

res_seq = RUNS[("secuencial", 1, 0)]["df"].sort_values("acc", ascending=False).reset_index(drop=True)
cols = [c for c in ["n_estimators", "max_depth", "max_features", "min_samples_split", "min_samples_leaf", "acc", "f1", "dur"] if c in res_seq.columns]
print("Top 10 combinaciones por accuracy (validación cruzada):")
display(res_seq[cols].head(10).style.format({"acc": "{:.4f}", "f1": "{:.4f}", "dur": "{:.1f}"}))
MEJOR = res_seq.iloc[0]

### 6.3 Métricas: media ± desviación estándar

Para cada repetición se calcula $S(p)=T_1/T_p$ (con $T_1$ = media de las corridas secuenciales) y $E(p)=S(p)/p$; luego se reporta media y desviación estándar sobre las repeticiones. Se incluyen el desbalance de carga, el tiempo ocioso y el tiempo ideal.

In [ ]:
T1 = np.mean([r["wall"] for (n, p, rep), r in RUNS.items() if n == "secuencial"])
filas = []
for (name, p, rep), r in RUNS.items():
    df, wall = r["df"], r["wall"]
    busy = df.groupby("worker")["dur"].sum()
    filas.append(dict(version=name, p=p, rep=rep, tiempo=wall, speedup=T1 / wall, eficiencia=(T1 / wall) / p,
                      desbalance=busy.max() / busy.mean(), t_ideal=df["dur"].sum() / p,
                      ocioso=100 * (1 - df["dur"].sum() / (p * wall))))
PER = pd.DataFrame(filas)
AGG = PER.groupby(["version", "p"]).agg(
    reps=("rep", "count"),
    tiempo_media=("tiempo", "mean"), tiempo_std=("tiempo", "std"),
    speedup_media=("speedup", "mean"), speedup_std=("speedup", "std"),
    efic_media=("eficiencia", "mean"), efic_std=("eficiencia", "std"),
    desbalance=("desbalance", "mean"), ocioso_pct=("ocioso", "mean"), t_ideal=("t_ideal", "mean")).reset_index().fillna(0)
RES.mkdir(exist_ok=True)
PER.to_csv(RES / "metricas_por_corrida.csv", index=False)
AGG.to_csv(RES / "resumen_metricas.csv", index=False)
display(AGG.style.format({"tiempo_media": "{:.1f}", "tiempo_std": "{:.1f}", "speedup_media": "{:.2f}", "speedup_std": "{:.2f}",
                          "efic_media": "{:.2f}", "efic_std": "{:.2f}", "desbalance": "{:.2f}", "ocioso_pct": "{:.0f}%", "t_ideal": "{:.1f}"}))

### 6.4 Gráficas: tiempo vs. número de procesos y *speedup* real vs. ideal

Las barras de error son la desviación estándar entre repeticiones. En la gráfica de *speedup* se añade la curva de Amdahl con la fracción paralela estimada para la mejor versión con el máximo de procesos.

In [ ]:
ORDEN = [("pool_estatico", "Pool estático", "#d1495b"), ("pool_dinamico", "Pool dinámico", "#00798c"),
         ("pool_dinamico_ordenado", "Pool dinámico ordenado", "#edae49"), ("mpi", "MPI maestro–trabajador", "#30638e"),
         ("ray", "Ray", "#6a994e")]
presentes = [(k, l, col) for k, l, col in ORDEN if k in AGG["version"].values]
ps = sorted(AGG["p"].unique())

def f_amdahl(S, p):
    return float(np.clip((1 - 1 / S) / (1 - 1 / p), 0, 1)) if p > 1 and S > 0 else np.nan

# fracción paralela de la mejor versión con PMAX (para dibujar la curva de Amdahl)
cand = AGG[(AGG["p"] == PMAX) & (AGG["version"].isin([k for k, _, _ in presentes]))]
f_best = f_amdahl(cand["speedup_media"].max(), PMAX) if len(cand) and PMAX > 1 else np.nan

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
seq = AGG[AGG["version"] == "secuencial"].iloc[0]
for k, lab, col in presentes:
    d = AGG[AGG["version"] == k]
    kw = dict(fmt="o-", capsize=3, label=lab, color=col)
    axes[0].errorbar(d["p"], d["tiempo_media"], yerr=d["tiempo_std"], **kw)
    axes[1].errorbar(d["p"], d["speedup_media"], yerr=d["speedup_std"], **kw)
    axes[2].errorbar(d["p"], d["efic_media"], yerr=d["efic_std"], **kw)
axes[0].axhline(seq["tiempo_media"], ls="--", color="gray", label="Secuencial (T₁)")
axes[0].set_title("Tiempo total vs. núcleos"); axes[0].set_ylabel("segundos")
pp = np.linspace(1, max(ps), 100)
axes[1].plot(pp, pp, "k--", label="Ideal  S = p")
if not np.isnan(f_best):
    axes[1].plot(pp, 1 / ((1 - f_best) + f_best / pp), ":", color="purple", label=f"Amdahl (f = {f_best:.2f})")
axes[1].set_title("Speedup real vs. ideal"); axes[1].set_ylabel("S(p) = T₁ / Tₚ")
axes[2].axhline(1, ls="--", color="k", label="Ideal"); axes[2].set_title("Eficiencia"); axes[2].set_ylabel("E(p) = S(p) / p"); axes[2].set_ylim(0, 1.2)
for a in axes:
    a.set_xlabel("procesos / núcleos (p)"); a.set_xticks(ps); a.legend(fontsize=8)
plt.tight_layout(); RES.mkdir(exist_ok=True); plt.savefig(RES / "escalamiento.png", bbox_inches="tight"); plt.show()

### 6.5 Comparación con `joblib` (máximo de procesos)

In [ ]:
jb = AGG[AGG["version"].str.startswith("joblib")]
if len(jb):
    cmp = pd.concat([jb, AGG[(AGG["p"] == PMAX) & (AGG["version"].isin(["pool_estatico", "pool_dinamico"]))]])
    display(cmp[["version", "p", "reps", "tiempo_media", "tiempo_std", "speedup_media", "efic_media", "desbalance"]].style.format(
        {"tiempo_media": "{:.1f}", "tiempo_std": "{:.1f}", "speedup_media": "{:.2f}", "efic_media": "{:.2f}", "desbalance": "{:.2f}"}))
else:
    print("No hay corridas de joblib.")

### 6.6 Ley de Amdahl: fracción paralela estimada

Con $f=\dfrac{1-1/S}{1-1/p}$ se estima qué fracción del trabajo se comporta como paralelizable a partir del *speedup* medido, y con $S_{\max}=1/(1-f)$ el límite teórico con infinitos procesos. Como $1-f$ absorbe **todo lo que no escala** (arranque de procesos, carga de datos, comunicación, tiempo ocioso por desbalance, ancho de banda de memoria), es una medida global del efecto de los cuellos de botella, no solo del código secuencial.

In [ ]:
filas = []
for _, r in AGG[(AGG["p"] > 1) & (AGG["version"] != "secuencial")].iterrows():
    f = f_amdahl(r["speedup_media"], r["p"])
    filas.append(dict(version=r["version"], p=int(r["p"]), speedup=r["speedup_media"], f_paralela=f,
                      fraccion_no_escala=1 - f, speedup_max_teorico=(1 / (1 - f)) if f < 1 else np.inf))
AMD = pd.DataFrame(filas)
AMD.to_csv(RES / "amdahl.csv", index=False)
display(AMD.style.format({"speedup": "{:.2f}", "f_paralela": "{:.3f}", "fraccion_no_escala": "{:.3f}", "speedup_max_teorico": "{:.1f}"}))

### 6.7 Reto extra: tiempo de cada tarea y efecto del desbalance

**a) ¿Qué tan distintas son las tareas?** Duración de cada combinación (corrida secuencial) según sus hiperparámetros.

In [ ]:
d = RUNS[("secuencial", 1, 0)]["df"].copy()
d["profundidad"] = d["max_depth"].fillna(40)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sc = axes[0].scatter(d["n_estimators"], d["dur"], c=d["profundidad"], cmap="viridis", s=40, edgecolor="k", linewidth=.3)
plt.colorbar(sc, ax=axes[0]).set_label("max_depth (None = 40)")
axes[0].set_xlabel("n_estimators"); axes[0].set_ylabel("duración de la tarea (s)"); axes[0].set_title("Costo de cada combinación")
axes[1].hist(d["dur"], bins=25, color="#4c72b0", edgecolor="white")
axes[1].set_xlabel("duración (s)"); axes[1].set_ylabel("número de tareas"); axes[1].set_title("Distribución de duraciones")
plt.tight_layout(); plt.savefig(RES / "duraciones.png", bbox_inches="tight"); plt.show()
print(f"Duración mínima {d['dur'].min():.2f} s · mediana {d['dur'].median():.2f} s · máxima {d['dur'].max():.2f} s · "
      f"razón máx/mín = {d['dur'].max()/d['dur'].min():.0f}x · coef. de variación = {d['dur'].std()/d['dur'].mean():.2f}")

**b) Línea de tiempo por trabajador (Gantt)**, primera repetición con el máximo de procesos. Cada fila es un trabajador y cada barra una tarea; los **huecos** son tiempo ocioso. Con reparto estático se ve cómo unos trabajadores terminan mucho antes que otros y el tiempo total lo marca el último.

In [ ]:
VMAX_DUR = max(r["df"]["dur"].max() for r in RUNS.values())

def gantt(ax, name, p, titulo):
    r = RUNS[(name, p, 0)]; df = r["df"]; cmap = plt.get_cmap("viridis")
    for _, t in df.iterrows():
        ax.broken_barh([(t["t_start"] - r["t0"], t["dur"])], (t["worker"] - 0.4, 0.8),
                       facecolors=cmap(float(min(t["dur"] / VMAX_DUR, 1.0))), edgecolor="white", linewidth=.3)
    ax.axvline(r["wall"], color="red", ls="--", lw=1)
    busy = df.groupby("worker")["dur"].sum()
    ax.set_title(f"{titulo}  (p={p})  ·  total {r['wall']:.1f} s  ·  desbalance {busy.max()/busy.mean():.2f}", fontsize=10)
    ax.set_ylabel("trabajador"); ax.set_yticks(range(p)); ax.invert_yaxis()

p_g = PMAX
mostrar = [(k, l) for k, l, _ in ORDEN if (k, p_g, 0) in RUNS]
fig, axes = plt.subplots(len(mostrar), 1, figsize=(12, 2.2 + 1.5 * len(mostrar) * max(1, min(p_g, 8) / 4)), sharex=True, layout="constrained")
axes = np.atleast_1d(axes)
for ax, (k, l) in zip(axes, mostrar):
    gantt(ax, k, p_g, l)
axes[-1].set_xlabel("segundos desde el inicio de la corrida")
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0, VMAX_DUR))
fig.colorbar(sm, ax=axes, label="duración de la tarea (s)", pad=0.01)
plt.savefig(RES / "gantt.png", bbox_inches="tight"); plt.show()

**c) Desbalance y tiempo ocioso** por estrategia y número de procesos (media de las repeticiones).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
ancho = 0.8 / max(1, len(presentes))
for i, (k, lab, col) in enumerate(presentes):
    dd = AGG[AGG["version"] == k].set_index("p").reindex(ps)
    x = np.arange(len(ps)) + i * ancho
    axes[0].bar(x, dd["desbalance"], ancho, label=lab, color=col)
    axes[1].bar(x, dd["ocioso_pct"], ancho, label=lab, color=col)
for a, t, yl in [(axes[0], "Desbalance de carga (máx / promedio)", "1.0 = perfecto"), (axes[1], "Tiempo ocioso total", "% del tiempo-procesador")]:
    a.set_xticks(np.arange(len(ps)) + ancho * (len(presentes) - 1) / 2); a.set_xticklabels(ps)
    a.set_xlabel("procesos (p)"); a.set_title(t); a.set_ylabel(yl)
axes[0].axhline(1, color="k", ls="--", lw=1); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RES / "desbalance.png", bbox_inches="tight"); plt.show()

### 6.8 Modelo final

Con la mejor combinación se reentrena sobre todo el conjunto de entrenamiento y se evalúa **una sola vez** en el conjunto de prueba (que no participó en la búsqueda). Como Covertype tiene clases desbalanceadas se reporta el F1 macro además de la *accuracy*.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, ConfusionMatrixDisplay

params = {k: MEJOR[k] for k in CFG["grid"].keys()}
params = {k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in params.items()}
for k in ("n_estimators", "min_samples_split", "min_samples_leaf", "max_depth"):
    if params.get(k) is not None: params[k] = int(params[k])
if isinstance(params.get("max_features"), str):
    try: params["max_features"] = float(params["max_features"])
    except ValueError: pass
print("Mejores hiperparámetros:", params)

final = RandomForestClassifier(random_state=SEMILLA, n_jobs=-1, **params).fit(X_tr, y_tr)
pred = final.predict(X_te)
print(f"\nPrueba -> accuracy = {accuracy_score(y_te, pred):.4f} · F1 macro = {f1_score(y_te, pred, average='macro'):.4f}\n")
print(classification_report(y_te, pred, digits=3))
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_te, pred, normalize="true", ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.grid(False); ax.set_title("Matriz de confusión normalizada (prueba)")
plt.show()

## 7. Análisis y discusión

### 7.1 Resumen automático de las mediciones

La siguiente celda redacta los hallazgos principales **a partir de las mediciones reales** de esta ejecución.

In [ ]:
def fila(k, p):
    r = AGG[(AGG["version"] == k) & (AGG["p"] == p)]
    return r.iloc[0] if len(r) else None

txt = [f"- **Línea base secuencial (T₁):** {seq['tiempo_media']:.1f} ± {seq['tiempo_std']:.1f} s ({seq['tiempo_media']/60:.1f} min) para {N_TAREAS} combinaciones × {CV_FOLDS} folds, "
       f"{int(seq['reps'])} repeticiones. La combinación más lenta tardó {d['dur'].max():.1f} s y la más rápida {d['dur'].min():.2f} s (razón {d['dur'].max()/d['dur'].min():.0f}x)."]
mejor_v, mejor_s = None, 0
for k, lab, _ in presentes:
    r = fila(k, PMAX)
    if r is not None and PMAX > 1:
        txt.append(f"- **{lab}, p={PMAX}:** {r['tiempo_media']:.1f} ± {r['tiempo_std']:.1f} s · speedup {r['speedup_media']:.2f} ± {r['speedup_std']:.2f} · "
                   f"eficiencia {r['efic_media']:.0%} · desbalance {r['desbalance']:.2f}.")
        if r["speedup_media"] > mejor_s: mejor_v, mejor_s = lab, r["speedup_media"]
if mejor_v: txt.append(f"- **Mejor versión con p={PMAX}:** {mejor_v}, con speedup de {mejor_s:.2f} (ideal: {PMAX}).")
est, din = fila("pool_estatico", PMAX), fila("pool_dinamico", PMAX)
if est is not None and din is not None and PMAX > 1:
    mejora = (est["tiempo_media"] - din["tiempo_media"]) / est["tiempo_media"] * 100
    if mejora > 0:
        txt.append(f"- **Estático vs. dinámico (p={PMAX}):** el reparto dinámico fue {mejora:.0f}% más rápido; el desbalance bajó de {est['desbalance']:.2f} a {din['desbalance']:.2f} "
                   f"y el tiempo ocioso de {est['ocioso_pct']:.0f}% a {din['ocioso_pct']:.0f}%.")
    else:
        txt.append(f"- **Estático vs. dinámico (p={PMAX}):** en esta ejecución no hubo ventaja del dinámico ({-mejora:.0f}% más lento); esto ocurre si la carga ya estaba "
                   f"bastante equilibrada o el costo de coordinación domina (desbalance estático {est['desbalance']:.2f}).")
orde = fila("pool_dinamico_ordenado", PMAX)
if orde is not None and din is not None and PMAX > 1:
    txt.append(f"- **Orden por costo estimado:** dinámico ordenado {orde['tiempo_media']:.1f} s vs. dinámico simple {din['tiempo_media']:.1f} s.")
if not np.isnan(f_best):
    txt.append(f"- **Ley de Amdahl:** con la mejor versión, la fracción paralela estimada es f ≈ {f_best:.2f}, lo que limita el speedup teórico a ≈ {1/(1-f_best) if f_best<1 else float('inf'):.1f}x aun con infinitos procesos.")
display(Markdown("\n".join(txt)))

### 7.2 Por qué mejora el rendimiento (y por qué no es perfecto)

**Por qué mejora.** Las combinaciones son independientes, así que *p* procesos evalúan *p* combinaciones a la vez. El tiempo total pasa de ser la **suma** de las duraciones de todas las tareas a ser, aproximadamente, esa suma dividida entre *p* (el tiempo ideal de la tabla).

**Por qué no se alcanza el *speedup* ideal** (cuellos de botella, a contrastar con las tablas y figuras de la sección 6):

1. **Desbalance de carga.** Con reparto estático el tiempo lo marca el trabajador que recibió el bloque más pesado; los demás esperan. Es visible en el Gantt, en el desbalance (> 1) y en el tiempo ocioso. El reparto dinámico reduce los huecos porque ningún proceso se queda sin trabajo mientras haya tareas; ordenar por costo evita que una tarea larga llegue al final, cuando ya no hay con qué compensarla.
2. **Secciones secuenciales o repetidas (Amdahl).** Crear procesos, importar scikit-learn y cargar el dataset en cada proceso no se acelera con más procesos; esa parte aparece como la fracción $1-f$. En MPI se suma el paso de mensajes y que el maestro ocupa un proceso (con *p* trabajadores se usan *p*+1).
3. **Recursos compartidos de hardware.** Con muchos procesos se comparten el ancho de banda de memoria y la caché (Random Forest recorre muchos datos); y si *p* supera los núcleos físicos (hiperhilos), el escalamiento deja de ser lineal.
4. **Granularidad.** Si las tareas fueran muy cortas, pedir trabajo con `chunksize=1` costaría más; aquí cada tarea dura decenas de milisegundos a segundos, y la coordinación pesa poco frente al entrenamiento (con dataset completo pesaría menos aún).

**Estático vs. dinámico: el compromiso.** El reparto estático no requiere comunicación durante la ejecución, pero depende de que las tareas cuesten parecido. El dinámico paga una pequeña coordinación por tarea a cambio de adaptarse a costos heterogéneos. Un `chunksize` intermedio es un punto medio.

**Pool vs. MPI.** `Pool` funciona dentro de una sola máquina; MPI está pensado para **múltiples nodos** (aquí corre en una sola, por lo que su ventaja es la escalabilidad futura y no el rendimiento local). El maestro–trabajador de MPI implementa de forma explícita el mismo reparto dinámico que `imap_unordered`.

### 7.3 Limitaciones

- Las pruebas se hicieron en **una sola máquina**; MPI no se evaluó en un clúster real, así que no se midió el costo de comunicación por red.
- Se usó una **muestra** de Covertype para que el experimento fuera ejecutable en tiempo razonable; con el dataset completo cada tarea sería más larga y la eficiencia tendería a mejorar porque la sobrecarga fija pesaría menos.
- Las repeticiones reducen el ruido, pero procesos en segundo plano, la temperatura/frecuencia de la CPU y el uso de hiperhilos pueden afectar las mediciones.
- El tiempo de MPI no incluye el arranque del intérprete ni el de `mpiexec`, mientras que el de `Pool` sí incluye la creación de procesos; la comparación directa debe leerse con esa salvedad.
- Los núcleos lógicos no equivalen a núcleos físicos: más allá del número de físicos se espera una caída de eficiencia.

## 8. Conclusiones

1. **Se paralelizó la búsqueda de hiperparámetros** con dos modelos de paralelismo distintos (procesos locales con `Pool`/`joblib` y memoria distribuida con MPI maestro–trabajador, más Ray opcional), obteniendo exactamente los mismos resultados que la versión secuencial.
2. **El rendimiento mejora** respecto al ciclo `for` secuencial (sección 6.3 y 6.4): el *speedup* crece con el número de procesos, con eficiencia menor a 1.
3. **El balance de carga es el factor decisivo.** Como el costo de una combinación varía en uno o más órdenes de magnitud, el reparto estático deja procesos ociosos; el dinámico (y el dinámico ordenado por costo) reduce el desbalance y acerca el tiempo al ideal.
4. **La ganancia está acotada** por lo que no escala, cuantificado con la fracción paralela de la Ley de Amdahl (sección 6.6).
5. **Recomendación práctica:** para búsquedas de hiperparámetros con tareas heterogéneas, usar reparto dinámico con `chunksize` pequeño y, si se puede estimar el costo, empezar por las tareas más pesadas.

> _Complementar esta sección con la interpretación del equipo usando las cifras exactas de las tablas de la sección 6._

## 9. Declaración de uso de herramientas de IA

> _Esta sección debe ser revisada y completada por el equipo con lo que realmente hizo cada integrante._

Se utilizó un asistente de IA (**Claude**, de Anthropic) como apoyo para:

- proponer el diseño experimental y la elección del dataset (Covertype) y del modelo (Random Forest);
- generar la primera versión del código (módulos del repositorio y este notebook);
- redactar explicaciones teóricas del informe.

Lo que el equipo hizo por su cuenta: _[ejecutar los experimentos en su equipo, revisar y modificar el código, interpretar los resultados, redactar las conclusiones, preparar la presentación… completar según corresponda]_.

Todos los integrantes deben poder explicar cualquier parte del código en la presentación.

## 10. Lista de verificación de requisitos obligatorios

| Requisito | Dónde se cumple |
|---|---|
| Versión secuencial base correcta | §5.2, `secuencial/secuencial.py` |
| Mínimo dos versiones paralelas con modelos distintos | §5.3 (Pool/joblib, procesos locales) y §5.4 (MPI, memoria distribuida); Ray opcional |
| Verificación de correctitud | §6.2 (mismos puntajes en todas las versiones y repeticiones) |
| Experimentos con 1, 2, 4, 8 y 16 procesos y ≥ 3 repeticiones; media y desviación estándar | §6.1 y §6.3 (`REPS`, `P_LIST`) |
| Métricas: tiempo, *speedup*, eficiencia y fracción paralela (Amdahl) | §6.3 y §6.6 |
| Gráficas de tiempo vs. núcleos y *speedup* real vs. ideal | §6.4 |
| Reproducibilidad: Dockerfile o requirements.txt + README | raíz del repositorio |
| Especificaciones del equipo de cómputo | §3 |
| Reto extra: tiempo por tarea y efecto del desbalance | §6.7 |
| Declaración de uso de IA | §9 |

## 11. Referencias

- Dua, D. y Graff, C. *Covertype Data Set*, UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/31/covertype
- scikit-learn: `ParameterGrid`. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ParameterGrid.html
- joblib: ejecución en paralelo. https://joblib.readthedocs.io/en/stable/parallel.html
- Python: `multiprocessing` (Pool, map, imap_unordered). https://docs.python.org/3/library/multiprocessing.html
- mpi4py. https://mpi4py.readthedocs.io
- Ray Tune (referencia de herramienta profesional). https://docs.ray.io/en/latest/tune/index.html